# 04 - Human-Supervised Topic Labels and Time Dynamics

                This notebook labels embedding-based clusters as candidate topics, writes evidence packs for manual review, compares unguided clusters with seed-topic lenses without making the seeds determinative, and exports the main topic-trend figures.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
assignments = pd.read_csv(ROOT / "outputs/tables/article_cluster_assignments_all_models.csv")
embedding_index = pd.read_csv(ROOT / "outputs/tables/article_embedding_index.csv")
embeddings = np.load(ROOT / "outputs/models/article_embeddings_main.npy")

topics_base = assignments[["article_id", "cluster_id_main", "cluster_solution_main", "embedding_row"]].merge(
    articles, on="article_id", how="left"
)
topics_base["cluster_id_main"] = topics_base["cluster_id_main"].astype(int)
print(topics_base.shape)

## Machine Labels and Manual Override Template

In [ ]:
label_rows = []
for cluster_id, group in topics_base.groupby("cluster_id_main"):
    terms = top_tfidf_terms(group["text_for_labeling"], n_terms=10)
    machine_label = " / ".join(terms[:4]) if terms else f"candidate topic {cluster_id}"
    label_rows.append(
        {
            "cluster_id_main": cluster_id,
            "topic_id": f"topic_{cluster_id}",
            "topic_label_machine": machine_label,
            "top_terms": "; ".join(terms),
            "n_articles": group["article_id"].nunique(),
            "first_year": group["year"].min(),
            "last_year": group["year"].max(),
            "topic_macro_category_machine": infer_macro_category(machine_label + " " + " ".join(terms)),
        }
    )

labels = pd.DataFrame(label_rows).sort_values(["cluster_id_main"])
override_path = ROOT / "outputs/labels/topic_label_overrides.csv"
if not override_path.exists():
    override_template = labels[["cluster_id_main", "topic_id", "topic_label_machine", "top_terms", "n_articles"]].copy()
    override_template["topic_label_human"] = ""
    override_template["topic_macro_category_human"] = ""
    override_template["label_status"] = "needs_human_review"
    override_template["label_rationale"] = ""
    write_csv(override_template, override_path)

overrides = pd.read_csv(override_path)
labels = labels.merge(
    overrides[["cluster_id_main", "topic_label_human", "topic_macro_category_human", "label_status", "label_rationale"]],
    on="cluster_id_main",
    how="left",
)
labels["topic_label_human"] = labels["topic_label_human"].map(compact_ws)
labels["topic_label_human"] = np.where(labels["topic_label_human"] != "", labels["topic_label_human"], labels["topic_label_machine"])
labels["topic_macro_category"] = labels["topic_macro_category_human"].map(compact_ws)
labels["topic_macro_category"] = np.where(
    labels["topic_macro_category"] != "",
    labels["topic_macro_category"],
    labels["topic_macro_category_machine"],
)
labels["label_status"] = labels["label_status"].fillna("machine_suggested")
labels["is_residual_topic"] = labels["cluster_id_main"].eq(-1) | labels["topic_label_human"].str.contains("residual|mixed|other", case=False, regex=True)

topic_stability = pd.read_csv(ROOT / "outputs/tables/topic_stability_metrics.csv")
main_solution = assignments["cluster_solution_main"].iloc[0]
main_stability_score = float(topic_stability.loc[topic_stability["cluster_solution_id"].eq(main_solution), "nmi_to_main"].fillna(1).iloc[0])
labels["topic_stability_score"] = main_stability_score

article_topics = topics_base[["article_id", "cluster_id_main"]].merge(labels, on="cluster_id_main", how="left")
article_topics["topic_probability_or_strength"] = np.nan
write_csv(labels, ROOT / "outputs/tables/topic_labels.csv")
write_csv(article_topics, ROOT / "outputs/tables/article_topics.csv")
display(labels.sort_values("n_articles", ascending=False).head(20))

## Topic Evidence Packs

In [ ]:
for cluster_id, group in topics_base.groupby("cluster_id_main"):
    label_row = labels[labels["cluster_id_main"].eq(cluster_id)].iloc[0]
    rows = group.sort_values("year")
    matrix = embeddings[rows["embedding_row"].to_numpy()]
    if len(rows) > 1:
        centroid = matrix.mean(axis=0, keepdims=True)
        sims = cosine_similarity(matrix, centroid).ravel()
        rows = rows.assign(_centroid_similarity=sims)
        representative = rows.sort_values("_centroid_similarity", ascending=False).head(15)
        marginal = rows.sort_values("_centroid_similarity", ascending=True).head(10)
    else:
        representative = rows
        marginal = rows

    period_dist = rows["period"].value_counts().to_string()
    language_dist = rows["language"].value_counts(dropna=False).to_string()
    type_dist = rows["type_en"].value_counts(dropna=False).head(10).to_string()
    author_counts = []
    for authors in rows["authors"].dropna():
        author_counts.extend(split_author_string(authors))
    author_summary = pd.Series(author_counts).value_counts().head(10).to_string() if author_counts else "No author data"
    issue_summary = rows["issue_title"].map(compact_ws).replace("", pd.NA).dropna().value_counts().head(10).to_string()
    if not issue_summary:
        issue_summary = "No recurring issue titles"

    pack = [
        f"# Evidence Pack: topic_{cluster_id}",
        "",
        f"- Machine label: {label_row['topic_label_machine']}",
        f"- Human label: {label_row['topic_label_human']}",
        f"- Macro category: {label_row['topic_macro_category']}",
        f"- Label status: {label_row['label_status']}",
        f"- Stability score inherited from main solution: {label_row['topic_stability_score']:.3f}",
        f"- Number of articles: {len(rows)}",
        "",
        "## Top Terms",
        "",
        label_row["top_terms"],
        "",
        "## Representative Titles",
        "",
    ]
    pack.extend(f"- {int(r.year)}: {compact_ws(r.title)}" for r in representative.itertuples())
    pack.extend(["", "## Marginal Titles", ""])
    pack.extend(f"- {int(r.year)}: {compact_ws(r.title)}" for r in marginal.itertuples())
    pack.extend(["", "## Period Distribution", "", "```", period_dist, "```"])
    pack.extend(["", "## Language Distribution", "", "```", language_dist, "```"])
    pack.extend(["", "## Document Types", "", "```", type_dist, "```"])
    pack.extend(["", "## Main Authors", "", "```", author_summary, "```"])
    pack.extend(["", "## Main Issues", "", "```", issue_summary, "```"])
    pack.extend(
        [
            "",
            "## Manual Interpretation",
            "",
            "Human label rationale: fill in `outputs/labels/topic_label_overrides.csv` and rerun this notebook.",
        ]
    )
    write_markdown("\n".join(pack), ROOT / f"outputs/diagnostics/topic_evidence/topic_{cluster_id}.md")

print(f"Wrote {labels.shape[0]} topic evidence packs.")

## Guided Seed Comparison as a Check, Not a Topic Model

In [ ]:
seed_topics = {
    "methods_and_methodology": ["method", "methods", "methodology", "qualitative", "quantitative", "grounded theory", "sampling"],
    "time_series_longitudinal": ["time series", "longitudinal", "panel", "sequence", "measurement", "classification"],
    "data_infrastructure": ["data", "database", "data archive", "coding", "digital history", "computing", "information system"],
    "digital_big_data_gis": ["big data", "computer", "digital", "gis", "geographic information", "algorithm"],
    "politics_elites_state": ["election", "voting", "party", "political", "elite", "crisis", "state"],
    "migration_borders_mobility": ["migration", "border", "migrant", "mobility", "regime"],
    "demography_family_population": ["family", "fertility", "demography", "population", "household"],
    "health_care_mortality": ["death", "mortality", "care", "health", "pandemic"],
    "sport_body_culture": ["sport", "football", "body", "leisure"],
    "conventions_law_economy": ["convention", "classification", "law", "economics of convention"],
    "environment_energy_sustainability": ["environment", "climate", "sustainability", "energy"],
    "science_technology_sts": ["science", "expertise", "knowledge", "technology", "sts"],
    "war_violence_security": ["war", "violence", "security", "risk", "genocide"],
    "culture_media_visuality": ["media", "visual", "image", "communication", "culture"],
}

seed_scores = topics_base[["article_id", "cluster_id_main", "text_for_labeling"]].copy()
text_lower = seed_scores["text_for_labeling"].map(lambda x: compact_ws(x).lower())
for seed_name, keywords in seed_topics.items():
    seed_scores[seed_name] = text_lower.map(lambda text: sum(text.count(keyword.lower()) for keyword in keywords))
seed_cols = list(seed_topics)
seed_scores["top_seed_topic"] = seed_scores[seed_cols].idxmax(axis=1)
seed_scores["top_seed_score"] = seed_scores[seed_cols].max(axis=1)

comparison = (
    seed_scores.groupby(["cluster_id_main", "top_seed_topic"])
    .agg(n_articles=("article_id", "nunique"), mean_seed_score=("top_seed_score", "mean"))
    .reset_index()
    .sort_values(["cluster_id_main", "n_articles"], ascending=[True, False])
)
comparison = comparison.merge(labels[["cluster_id_main", "topic_id", "topic_label_human"]], on="cluster_id_main", how="left")
write_csv(comparison, ROOT / "outputs/tables/guided_vs_unguided_topic_comparison.csv")
display(comparison.groupby("cluster_id_main").head(3))

## Topic Trends, Sensitivity, and Innovation Metrics

In [ ]:
topic_articles = articles.merge(article_topics[["article_id", "topic_id", "topic_label_human", "topic_macro_category", "topic_stability_score"]], on="article_id", how="inner")

def topic_shares(df, group_col):
    counts = df.groupby([group_col, "topic_id", "topic_label_human", "topic_macro_category"], dropna=False).size().reset_index(name="n_articles")
    totals = df.groupby(group_col).size().rename("n_total").reset_index()
    out = counts.merge(totals, on=group_col)
    out["topic_share"] = out["n_articles"] / out["n_total"]
    return out

year_trends = topic_shares(topic_articles, "year")
year_trends = year_trends.sort_values(["topic_id", "year"])
year_trends["rolling_share_5y"] = year_trends.groupby("topic_id")["topic_share"].transform(lambda s: s.rolling(5, center=True, min_periods=1).mean())
period_trends = topic_shares(topic_articles, "period")

write_csv(year_trends, ROOT / "outputs/tables/topic_trends_by_year.csv")
write_csv(period_trends, ROOT / "outputs/tables/topic_trends_by_period.csv")

def summarize_condition(name, mask):
    subset = topic_articles[mask].copy()
    if subset.empty:
        return pd.DataFrame()
    out = topic_shares(subset, "period")
    out["condition"] = name
    return out

sensitivity = pd.concat(
    [
        summarize_condition("core", topic_articles["corpus_inclusion_core"].fillna(False)),
        summarize_condition("extended", topic_articles["corpus_inclusion_extended"].fillna(False)),
        summarize_condition("core_without_review_editorial_biblio", topic_articles["corpus_inclusion_core"].fillna(False) & ~topic_articles[["is_review", "is_editorial_or_intro", "is_bibliography"]].any(axis=1)),
    ],
    ignore_index=True,
)
write_csv(sensitivity, ROOT / "outputs/tables/topic_trend_sensitivity.csv")

macro_year = topic_articles.groupby(["year", "topic_macro_category"]).size().reset_index(name="n_articles")
macro_pivot = macro_year.pivot_table(index="year", columns="topic_macro_category", values="n_articles", fill_value=0)
fig, ax = plt.subplots(figsize=(12, 5))
macro_pivot.plot.area(ax=ax, alpha=0.85)
ax.set_title("Macro topic streams")
ax.set_xlabel("Year")
ax.set_ylabel("Number of articles")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Macro category")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_05_macro_topic_streams.png", dpi=220)
save_caption(ROOT, "fig_05_macro_topic_streams.png", "Changing distribution of broad thematic fields in HSR; categories are human-supervised labels on model-induced clusters.")
plt.show()

heatmap = period_trends.pivot_table(index="topic_label_human", columns="period", values="topic_share", fill_value=0)
fig, ax = plt.subplots(figsize=(12, max(6, 0.28 * len(heatmap))))
sns.heatmap(heatmap, cmap="mako", ax=ax)
ax.set_title("Topic prominence by period")
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_06_topic_period_heatmap.png", dpi=220)
save_caption(ROOT, "fig_06_topic_period_heatmap.png", "Period heatmap of candidate topic shares; only stable and reviewed topics should be emphasized in the main text.")
plt.show()

top_topics = topic_articles["topic_label_human"].value_counts().head(8).index
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=year_trends[year_trends["topic_label_human"].isin(top_topics)], x="year", y="rolling_share_5y", hue="topic_label_human", ax=ax)
ax.set_title("Selected topic trajectories")
ax.set_xlabel("Year")
ax.set_ylabel("Five-year rolling share")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Topic")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_07_selected_topic_trajectories.png", dpi=220)
save_caption(ROOT, "fig_07_selected_topic_trajectories.png", "Five-year rolling shares for the most frequent candidate topics.")
plt.show()

innovation_rows = []
for topic_id, group in topic_articles.groupby("topic_id"):
    by_year = group.groupby("year").size()
    first_year = int(by_year.index.min())
    peak_year = int(by_year.idxmax())
    active_years = set(by_year.index.astype(int))
    possible_years = set(range(first_year, int(topic_articles["year"].max()) + 1))
    persistence = len(active_years & possible_years) / max(len(possible_years), 1)
    top_issues = group["issue_id"].value_counts().head(2).sum()
    issue_dependency = top_issues / len(group)
    innovation_rows.append(
        {
            "topic_id": topic_id,
            "topic_label_human": group["topic_label_human"].iloc[0],
            "first_year": first_year,
            "first_period": group.loc[group["year"].eq(first_year), "period"].iloc[0],
            "peak_year": peak_year,
            "peak_share": float(year_trends.loc[(year_trends["topic_id"].eq(topic_id)) & (year_trends["year"].eq(peak_year)), "topic_share"].max()),
            "n_years_active": len(active_years),
            "persistence_score": persistence,
            "issue_dependency_score": issue_dependency,
            "is_stable_core": persistence >= 0.25 and issue_dependency < 0.55 and group["period"].nunique() >= 3,
            "is_emerging_topic": first_year >= 2010 and persistence >= 0.10,
            "is_declining_topic": peak_year < 2005 and persistence < 0.35,
            "is_episode_topic": issue_dependency >= 0.55 and persistence < 0.25,
        }
    )
innovation = pd.DataFrame(innovation_rows).sort_values(["first_year", "topic_label_human"])
write_csv(innovation, ROOT / "outputs/tables/topic_innovation_metrics.csv")

fig, ax = plt.subplots(figsize=(12, max(5, 0.25 * len(innovation))))
plot_innov = innovation.sort_values("first_year")
ax.scatter(plot_innov["first_year"], plot_innov["topic_label_human"], s=(plot_innov["persistence_score"] * 350 + 20), c=plot_innov["issue_dependency_score"], cmap="viridis", alpha=0.8)
ax.set_title("Topic entry and persistence")
ax.set_xlabel("First year")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_08_topic_entry_and_persistence_timeline.png", dpi=220)
save_caption(ROOT, "fig_08_topic_entry_and_persistence_timeline.png", "First appearance of candidate topics; marker size indicates persistence and color indicates issue dependency.")
plt.show()

surprise = topics_base.merge(article_topics[["article_id", "topic_label_human"]], on="article_id", how="left")
surprise["expected_topic"] = ""
surprise["assigned_topic"] = surprise["topic_label_human"]
surprise["reason_for_surprise"] = ""
surprise["manual_comment"] = ""
write_csv(surprise[["article_id", "title", "year", "authors", "expected_topic", "assigned_topic", "reason_for_surprise", "manual_comment"]].head(100), ROOT / "outputs/tables/surprise_cases.csv")

try:
    import plotly.express as px

    umap_main = pd.read_csv(ROOT / "outputs/tables/article_umap_main.csv")
    interactive = umap_main.merge(
        topic_articles[
            [
                "article_id",
                "authors",
                "issue_title",
                "type_en",
                "topic_label_human",
                "topic_macro_category",
            ]
        ],
        on="article_id",
        how="left",
    )
    interactive["issue_title"] = interactive["issue_title"].map(compact_ws)
    fig = px.scatter(
        interactive,
        x="umap_x",
        y="umap_y",
        color="topic_label_human",
        hover_data={
            "title": True,
            "year": True,
            "authors": True,
            "issue_title": True,
            "language": True,
            "type_en": True,
            "topic_macro_category": True,
            "umap_x": False,
            "umap_y": False,
        },
        title="Interactive HSR Semantic Map",
        width=1100,
        height=800,
    )
    fig.update_traces(marker={"size": 6, "opacity": 0.75})
    fig.write_html(ROOT / "outputs/figures/interactive_hsr_map.html", include_plotlyjs="cdn")
    print("Wrote outputs/figures/interactive_hsr_map.html")
except Exception as exc:
    print(f"Interactive map skipped: {exc}")
display(innovation.head(20))